In [2]:
path_to_data = "/Volumes/Parler/"

In [3]:
from IPython.display import clear_output
import json
import os
import pandas as pd
from tqdm.auto import tqdm
import tarfile


Types of content from DDOS:
- Images
- Videos
- Posts (HTML)

Types of content from Zenodo:
- Users


In [13]:
# Get names for located files

files_df = pd.read_csv('../data/outputs/processed_files_GPS.txt', header = None)
files_df.columns=['filename', 'report']
files_df.head()

,filename,report
0,metadata/meta-00CnBY5xCdca.json,GPS_not_detected
1,metadata/meta-0003lx5cSwSB.json,GPS_not_detected
2,metadata/meta-0070HNolzi3z.json,GPS_not_detected
3,metadata/meta-00BIFOMnOyi1.json,GPS_not_detected
4,metadata/meta-0002bz1GNsUP.json,GPS_not_detected


In [20]:
target_files = files_df[files_df['report'].str.contains('GPS_detected')]
target_files.head()
target_list = target_files['filename'].tolist()


In [16]:
# Preview metadata.tar.gz 

metadata_stream = tarfile.open(os.path.join(path_to_data, "data","metadata.tar.gz"), 'r:gz')

In [22]:
target_list[3]

'metadata/meta-006OZ1YYlTnM.json'

In [29]:
# Preview metadata files

def extract_json(filename):
    contentObject = metadata_stream.extractfile(filename)

    # Add encoding step as the files original encoded in ascii not utf-8
    jsonContent = contentObject.read().decode('ascii')

    # print(jsonContent)
    return jsonContent

In [30]:
jsonContent = extract_json(target_list[3])
pd.read_json(jsonContent)

,SourceFile,ExifToolVersion,FileType,FileTypeExtension,MIMEType,MajorBrand,MinorVersion,CompatibleBrands,MediaDataSize,MediaDataOffset,...,CreationDate,ImageSize,Megapixels,AvgBitrate,GPSAltitude,GPSAltitudeRef,GPSLatitude,GPSLongitude,Rotation,GPSPosition
0,-,12,MOV,mov,video/quicktime,Apple QuickTime (.MOV/QT),0.0.0,[qt ],117566575,36,...,2021:01:04 18:59:50-05:00,1920x1080,2.1,13.9 Mbps,109.296 m,Above Sea Level,"38 deg 55' 58.80"" N","77 deg 4' 27.48"" W",90,"38 deg 55' 58.80"" N, 77 deg 4' 27.48"" W"


In [62]:
pbar = tqdm(desc='Extracting Video Metadata', total=len(target_list))

metadata = pd.DataFrame()
for each in target_list:
    jsonContent = extract_json(each)
    df = pd.read_json(jsonContent)
    metadata = pd.concat([metadata, df])
    pbar.update(1)


Extracting Video Metadata:   0%|          | 0/68588 [00:00<?, ?it/s]

In [63]:
metadata.head()

,SourceFile,ExifToolVersion,FileType,FileTypeExtension,MIMEType,MajorBrand,MinorVersion,CompatibleBrands,MediaDataSize,MediaDataOffset,...,SoftwareVersion-jpn,ContentCreateDate-jpn,GPSCoordinates-jpn,Model-jpn,Make-jpn,CreationDate-jpn-JP,Model-jpn-JP,Software-jpn-JP,GPSCoordinates-jpn-JP,Make-jpn-JP
0,-,12,MOV,mov,video/quicktime,Apple QuickTime (.MOV/QT),0.0.0,[qt ],39803620.0,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,-,12,MOV,mov,video/quicktime,Apple QuickTime (.MOV/QT),0.0.0,[qt ],12270766.0,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,-,12,MOV,mov,video/quicktime,Apple QuickTime (.MOV/QT),0.0.0,[qt ],21197396.0,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,-,12,MOV,mov,video/quicktime,Apple QuickTime (.MOV/QT),0.0.0,[qt ],117566575.0,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,-,12,MOV,mov,video/quicktime,Apple QuickTime (.MOV/QT),0.0.0,[qt ],51613271.0,36.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
metadata.to_csv('../data/outputs/located_metdata.csv', index=False)